# Artefacts in ODYSSEY (H2)

H2 (#19). The methods come from the literature collected in `docs/method.md`, *Cleaning of ODYSSEY (H2)*;
the review of #74 chose the EURADCLIM filters that run on the composite alone (Gabella, `histo_cut` on past
composites), a threshold for impossible values, and Wagner-style frequency maps for the catalogue.

1. **Gabella on rain** (review of #74, point 5): how much convective rain the filter removes on summer
   convective days, at 2 km (ODYSSEY grid) and at 1 km (CIRRUS), and where the removed rain lies.
2. **Gabella on the January features**: how much it removes on the January 2025 days whose `lk` deficit
   came from stationary features centred on radars (H1, `docs/method.md`).
3. **Pixels above 55 and 70 dBZ** on ODYSSEY and CIRRUS (review of #74, impossible values).
4. **Catalogue:** frequency maps of reflectivity classes and `histo_cut` on annual sums, per year.
5. **Static clutter across years:** how stable the `histo_cut` mask is from one year to the next.
6. **First version on ODYSSEY 2023:** the three steps of the priority in *Choice* (> 70 dBZ, Gabella,
   `histo_cut` of the previous year), with numbers and maps before and after.

In [ ]:
from opera_ds.paths import DATA_DIR, RESULTS_DIR

import csv
import glob
from collections import defaultdict

import numpy as np
from scipy import ndimage
from wradlib.classify import filter_gabella

from opera_ds.classify import obs_classes
from opera_ds.odim_io import index_files, read_odim_composite
from opera_ds.validate import read_degraded

## Settings

- **Data:** CIRRUS over the Belgian crop at :00, :15, :30 and :45. At 2 km it is degraded by `zmean`, the H1
  stand-in for ODYSSEY; at 1 km it is the real CIRRUS.
- **Filter:** `wradlib.classify.filter_gabella` with its defaults (5 × 5 window, `tr1` = 6 dB, `n_p` = 6,
  `tr2` = 1.3), `cartesian=True`, as in EURADCLIM (Overeem et al. 2023, section 3.1). Field in dBZ,
  `undetect` and values below 0 dBZ set to 0 dBZ (`thrsnorain` = 0), NaN outside coverage.
- **Classes:** Steiner on the 2 km grid, as in H1; at 1 km each 2 km class is copied to its 2 × 2 block.
- **Rain:** Marshall–Palmer, dBZ capped at 55 (Imhoff et al. 2020), on echoes ≥ 10 dBZ (the wet threshold of
  `run_validation`).
- **Where the removed rain lies:** each removed convective pixel is put in one of three groups.
  - *small echo:* its echo region (echoes above 0 dBZ, 8-connected) has at most 11 pixels, the size below
    which the area/perimeter test removes a region (Gabella and Notarpietro 2002, section 2);
  - *edge:* otherwise, when one of its 8 neighbours has no echo;
  - *interior:* the rest.

In [ ]:
WET_DBZ = 10.0
SMALL_PX = 11
EIGHT = np.ones((3, 3), bool)

files = index_files(str(DATA_DIR / "cirrus_bel" / "**" / "*@DBZH.h5"), r"@(\d{8}T\d{4})@", "%Y%m%dT%H%M")


def rain(dbz):
    return (10 ** (np.minimum(dbz, 55.0) / 10) / 200.0) ** (1 / 1.6)


def analyse(path, km):
    """Sums of rain and pixels, flagged or not, per class, and of removed convective rain per position."""
    if km == 2:
        f = read_degraded(path, factor=2).to_field(0.0)
        cls = obs_classes(f, floor=0.0, factor=1, pixel_km=2.0)
    else:
        f = read_odim_composite(path, "DBZH").to_field(0.0)
        cls = obs_classes(f, floor=0.0, factor=2, pixel_km=1.0)
    flag = filter_gabella(f.copy(), wsize=5, cartesian=True)
    echo = np.isfinite(f) & (f > 0)
    wet = np.isfinite(f) & (f >= WET_DBZ)
    r = np.where(wet, rain(np.nan_to_num(f)), 0.0)
    out = {}
    for name, m in (("all", wet), ("convective", wet & cls["convective"]), ("stratiform", wet & cls["stratiform"])):
        out[name] = np.array([r[m].sum(), r[m & flag].sum(), m.sum(), (m & flag).sum()], float)
    lab, n = ndimage.label(echo, structure=EIGHT)
    size = np.bincount(lab.ravel())
    small = echo & (size[lab] <= SMALL_PX)
    edge = echo & ~small & (ndimage.minimum_filter(echo.astype(np.uint8), footprint=EIGHT, mode="constant", cval=1) == 0)
    rem = wet & cls["convective"] & flag
    out["position"] = np.array([r[rem & small].sum(), r[rem & edge].sum(), r[rem & ~small & ~edge].sum()])
    return out


def run(days):
    tot = {km: defaultdict(lambda: defaultdict(float)) for km in (2, 1)}
    todo = [(f"{t:%Y-%m-%d}", str(p)) for t, p in sorted(files.items()) if t.minute % 15 == 0 and f"{t:%Y-%m-%d}" in days]
    print(len(todo), "frames")
    for km in (2, 1):
        for d, path in todo:
            for k, v in analyse(path, km).items():
                for key in (d, "all days"):
                    tot[km][key][k] = tot[km][key][k] + v
    return tot


def report(tot, days):
    for km in (2, 1):
        print(f"--- {km} km")
        print(f"{'day':12s} {'class':11s} {'rain removed':>13s} {'pixels removed':>15s}")
        for d in list(days) + ["all days"]:
            for k in ("all", "convective", "stratiform"):
                s, sr, n, nr = tot[km][d][k]
                print(f"{d:12s} {k:11s} {sr / s:12.1%} {nr / n:14.1%}")
        p = tot[km]["all days"]["position"]
        print("removed convective rain: small echo {:.0%}, edge {:.0%}, interior {:.0%}".format(*(p / p.sum())))

## 1. Summer convective days

The 5 days of June–July 2025 with most Steiner convective pixels in the H1 run (rows at `alpha` = 0, which
are the same for every method).

In [ ]:
days = defaultdict(int)
for mon in ("20250601-20250701", "20250701-20250801"):
    with open(RESULTS_DIR / "h1_degraded_cirrus" / mon / "per_time.csv", encoding="utf-8") as fh:
        for r in csv.DictReader(fh):
            if r["method"] == "nearest|convective" and float(r["alpha"]) == 0.0:
                days[r["time"][:10]] += int(r["n_wet"])
summer = sorted(days, key=lambda d: -days[d])[:5]
print("days (convective pixels, H1, alpha = 0):", [(d, days[d]) for d in summer])
tot_summer = run(summer)
report(tot_summer, summer)

## 2. January days with stationary features

The 5 days that carry 70% of the January `lk` deficit on convective pixels, with stationary features centred
on or near radars (`notebooks/03-mr-cirrus-analysis.ipynb`, section 4). Here the "convective" pixels are
mostly those features, so what the filter removes is a gain, not a loss.

In [ ]:
with open(RESULTS_DIR / "cirrus_analysis" / "selected_lk_deficit_times.csv", encoding="utf-8") as fh:
    january = [r["time"][:10] for r in csv.DictReader(fh)]
print("days:", january)
tot_january = run(january)
report(tot_january, january)

## 3. Pixels above 55 and 70 dBZ

Echo pixels (covered, not `undetect`) of the Belgian crops on the project PC: ODYSSEY 2023 and CIRRUS 2025.
About 40 min in one process for CIRRUS; the counts of the run of 9/11/2026 are in `docs/method.md`,
*Impossible values: counts on ODYSSEY and CIRRUS*. Set `RUN = True` to recount.

In [ ]:
RUN = False


def count(path):
    fr = read_odim_composite(path, "DBZH")
    v = fr.values[fr.coverage & ~fr.undetect]
    return np.array([v.size, ((v >= 55) & (v <= 70)).sum(), (v > 70).sum(), (v >= 100).sum(), int((v > 70).any()), 1])


if RUN:
    for name, pat in (("ODYSSEY 2023", "odyssey_bel/2023/**/*.h5"), ("CIRRUS 2025", "cirrus_bel/2025/**/*@DBZH.h5")):
        tot = sum(count(p) for p in sorted(glob.glob(str(DATA_DIR / pat), recursive=True)))
        n, a, b, c, fa, nf = tot
        print(f"{name}: {nf} files, {n:,} echo pixels; 55-70 dBZ {a:,} ({a / n:.2e}); >70 dBZ {b:,} ({b / n:.2e}), "
              f"of which >=100 dBZ {c:,}; files with any >70: {fa} ({fa / nf:.1%})")

## 4. Catalogue: frequency maps and `histo_cut`, per year

The catalogue of the review of #74, on ODYSSEY 2013–2024 and CIRRUS 2025 (`RUN_YEARS`; 2012 is left out, it
has 576 readable frames, `docs/data.md`, *Known problems*). Maps are drawn for ODYSSEY 2023 and CIRRUS 2025
(`PLOT_YEARS`); every year is printed. One pass per product and year stores, for each pixel:

- **Frequency maps** (Wagner et al. 2012, section 3, Table 1): how often the pixel falls in each reflectivity
  class: 7–19, 19–28, 28–37, 37–46, 46–55 and ≥ 55 dBZ. The paper uses 1 dBZ instead of 7 in winter; 7 is
  kept all year here.
- **Annual sum** of the rain rate (Marshall–Palmer, capped at 55 dBZ; 15 min steps for ODYSSEY, 5 min for
  CIRRUS), and the number of valid times.
- **Times above 70 dBZ**, the discard threshold proposed in the review of #74.

Then `wradlib.classify.histo_cut` on the annual sum, with the 5% tails of EURADCLIM (Overeem et al. 2023,
section 3.3) instead of the 1% default: class 1 = persistently high (static clutter), class 2 = persistently
low (shading, blockage). Under the real-time rule the mask of one year is to be applied to the following
years, not to its own; here it is only mapped.

Results are cached in `RESULTS_DIR / "h2_artefacts" / "catalogue_<product>_<year>.npz"`; one ODYSSEY year
takes about 10 min (six years at once on the project PC, 10/11/2026), one CIRRUS year about 40 min.

In [ ]:
from wradlib.classify import histo_cut

CLASS_EDGES = [7, 19, 28, 37, 46, 55]     # Wagner et al. 2012, Table 1 (lower edges, dBZ)
HISTO_FREQ = 0.05                         # Overeem et al. 2023, section 3.3
CAT = RESULTS_DIR / "h2_artefacts"
CAT.mkdir(parents=True, exist_ok=True)
PRODUCTS = {"odyssey": ("odyssey_bel", "*.h5", 0.25), "cirrus": ("cirrus_bel", "*@DBZH.h5", 5 / 60)}
RUN_YEARS = {"odyssey": list(range(2013, 2025)), "cirrus": [2025]}   # 2012 has 576 frames (data.md, Known problems)
PLOT_YEARS = {("odyssey", 2023), ("cirrus", 2025)}


def static_clutter(c):
    """histo_cut on the annual sum of the pixels with at least one valid time (wradlib's histo_cut fails
    when the input has NaN, as for the 337 pixels never covered in 2013)."""
    ok = c["valid"] > 0
    mask = np.zeros(ok.shape, np.uint8)
    mask[ok] = histo_cut(c["annual_sum"][ok], upper_frequency=HISTO_FREQ, lower_frequency=HISTO_FREQ)
    return mask


def catalogue(product, year):
    path = CAT / f"catalogue_{product}_{year}.npz"
    if path.exists():
        return dict(np.load(path))
    folder, pattern, step_h = PRODUCTS[product]
    files = sorted(glob.glob(str(DATA_DIR / folder / str(year) / "**" / pattern), recursive=True))
    freq = acc = valid = gt70 = None
    for p in files:
        fr = read_odim_composite(p, "DBZH")
        ok = fr.coverage
        v = np.where(ok & ~fr.undetect, fr.values, -np.inf)
        if freq is None:
            freq = np.zeros((len(CLASS_EDGES),) + v.shape, np.int32)
            acc, valid, gt70 = np.zeros(v.shape), np.zeros(v.shape, np.int32), np.zeros(v.shape, np.int32)
        cls = np.digitize(v, CLASS_EDGES) - 1          # -1 below 7 dBZ
        for k in range(len(CLASS_EDGES)):
            freq[k] += cls == k
        acc += np.where(np.isfinite(v), rain(np.where(np.isfinite(v), v, 0.0)), 0.0) * step_h
        valid += ok
        gt70 += v > 70
    out = {"freq": freq, "annual_sum": acc, "valid": valid, "gt70": gt70, "n_files": np.array(len(files))}
    np.savez_compressed(path, **out)
    return out


cat = {}
for product, years in RUN_YEARS.items():
    for year in years:
        c = catalogue(product, year)
        mask = static_clutter(c)
        cat[(product, year)] = (c, mask)
        print(f"{product} {year}: {int(c['n_files'])} files; annual sum median {np.median(c['annual_sum'][c['valid'] > 0]):.0f} mm, "
              f"max {c['annual_sum'].max():.0f} mm; histo_cut static clutter {int((mask == 1).sum())} pixels, "
              f"shading {int((mask == 2).sum())} pixels; pixels above 70 dBZ at least once {int((c['gt70'] > 0).sum())}")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

for (product, year), (c, mask) in cat.items():
    if (product, year) not in PLOT_YEARS:
        continue
    fig, ax = plt.subplots(1, 4, figsize=(22, 5.5), layout="constrained")
    im = ax[0].imshow(c["annual_sum"], cmap="viridis", vmax=np.percentile(c["annual_sum"], 99.5))
    ax[0].set_title(f"{product} {year}: annual sum (mm)"); fig.colorbar(im, ax=ax[0], shrink=0.8)
    im = ax[1].imshow(c["freq"][0] / np.maximum(c["valid"], 1), cmap="magma_r")
    ax[1].set_title("frequency of 7-19 dBZ"); fig.colorbar(im, ax=ax[1], shrink=0.8)
    im = ax[2].imshow(c["freq"][-1] / np.maximum(c["valid"], 1), cmap="magma_r", norm=LogNorm(vmin=1e-5, vmax=1))
    ax[2].set_title("frequency of >= 55 dBZ"); fig.colorbar(im, ax=ax[2], shrink=0.8)
    ax[3].imshow(np.where(mask > 0, mask, np.nan), cmap="coolwarm", vmin=1, vmax=2, interpolation="nearest")
    ax[3].set_title("histo_cut: static clutter (blue), shading (red)")
    for a_ in ax:
        a_.set_xticks([]); a_.set_yticks([])
    plt.show()

## 5. Static clutter across years

Under the real-time rule the `histo_cut` mask of one year is applied to the next. This only works if the
mask is stable: for each pixel, the number of ODYSSEY years (2013–2024) in which it is flagged, and the share
of each year's mask that was already flagged the year before.

In [ ]:
years = sorted(y for p, y in cat if p == "odyssey")
flagged = np.array([cat[("odyssey", y)][1] == 1 for y in years])
n_years = flagged.sum(0)
print(f"flagged at least once {int((n_years > 0).sum()):,} px; in all {len(years)} years {int((n_years == len(years)).sum()):,}; "
      f"in 10 or more {int((n_years >= 10).sum()):,}; in 1-3 years {int(((n_years > 0) & (n_years <= 3)).sum()):,}")
for a, b in zip(years[:-1], years[1:]):
    pa, pb = flagged[years.index(a)], flagged[years.index(b)]
    print(f"{b}: {int(pb.sum()):6,} px flagged, {(pa & pb).sum() / pb.sum():.0%} already flagged in {a}")

fig, ax = plt.subplots(1, 2, figsize=(13, 5.5), layout="constrained")
im = ax[0].imshow(np.where(n_years > 0, n_years, np.nan), cmap="viridis", vmin=1, vmax=len(years), interpolation="nearest")
ax[0].set_title(f"histo_cut static clutter: years flagged (of {len(years)})"); fig.colorbar(im, ax=ax[0], shrink=0.8)
c19 = cat[("odyssey", 2019)][0]["annual_sum"]
im = ax[1].imshow(c19, cmap="viridis", vmax=np.percentile(c19, 99.5)); ax[1].set_title("ODYSSEY 2019: annual sum (index, mm)")
fig.colorbar(im, ax=ax[1], shrink=0.8)
for a_ in ax:
    a_.set_xticks([]); a_.set_yticks([])
plt.show()

## 6. First version on ODYSSEY 2023

The priority of *Choice* (`docs/method.md`), applied to each 15 min composite of 2023 in this order:

1. **Impossible values:** > 70 dBZ (Fluck et al. 2021, the convention in *Choice*) set to no rain.
2. **Gabella:** `filter_gabella` as in section 1; flagged cells set to no rain, as in EURADCLIM (Overeem et
   al. 2023, section 3.1).
3. **Static mask:** `histo_cut` (5% tails) of **2022**, the previous year, as the real-time rule requires.
   The rain of a masked cell is replaced by the inverse-distance mean (power 2) of its 4 nearest unmasked
   cells, only when that is lower (EURADCLIM, section 3.3, which does it on hourly sums; here on each
   composite).

Rain: Marshall–Palmer, capped at 55 dBZ, echoes ≥ 7 dBZ (Imhoff et al. 2020). Classes: Steiner on the
original field at 2 km, as in section 1. Belgium: the Natural Earth mask of `data/mask_BEL_1km.npy` on the
2 km crop (cells at least half inside). Sequential, about 40 min; cached in
`RESULTS_DIR / "h2_artefacts" / "first_version_odyssey_2023.npz`.

In [ ]:
from scipy.spatial import cKDTree

FV_YEAR = 2023
FV_STEP_H = 0.25
HIST_EDGES = np.arange(-10, 131, 1.0)
FV_CLASSES = ("all", "convective", "stratiform")


def rain7(dbz):
    return np.where(dbz >= 7, rain(dbz), 0.0)


def first_version(year):
    path = CAT / f"first_version_odyssey_{year}.npz"
    if path.exists():
        return dict(np.load(path))
    mask = static_clutter(catalogue("odyssey", year - 1)) == 1
    good, bad = np.argwhere(~mask), np.argwhere(mask)
    d, i = cKDTree(good).query(bad, k=4)
    nb, w = good[i], 1.0 / d ** 2
    w /= w.sum(1, keepdims=True)
    acc = {k: np.zeros(mask.shape) for k in ("before", "after70", "afterGab", "after")}
    f55 = {k: np.zeros(mask.shape, np.int32) for k in ("before", "after")}
    hist = {k: np.zeros(len(HIST_EDGES) - 1) for k in ("before", "after")}
    removed = {c: np.zeros(4) for c in FV_CLASSES}   # rain before, removed by >70, by Gabella, by the static mask
    files = sorted(glob.glob(str(DATA_DIR / "odyssey_bel" / str(year) / "**" / "*.h5"), recursive=True))
    for p in files:
        f = read_odim_composite(p, "DBZH").to_field(0.0)
        cls = obs_classes(f, floor=0.0, factor=1, pixel_km=2.0)
        r0 = rain7(np.nan_to_num(f))
        z1 = np.where(f > 70, 0.0, f)
        r1 = rain7(np.nan_to_num(z1))
        z2 = np.where(filter_gabella(z1.copy(), wsize=5, cartesian=True), 0.0, z1)
        r2 = rain7(np.nan_to_num(z2))
        r3 = r2.copy()
        r3[bad[:, 0], bad[:, 1]] = np.minimum(r2[bad[:, 0], bad[:, 1]], (r2[nb[..., 0], nb[..., 1]] * w).sum(1))
        z3 = np.where(r3 < r2, 0.0, z2)
        for k, r in (("before", r0), ("after70", r1), ("afterGab", r2), ("after", r3)):
            acc[k] += r * FV_STEP_H
        f55["before"] += np.nan_to_num(f) >= 55
        f55["after"] += np.nan_to_num(z3) >= 55
        hist["before"] += np.histogram(f[np.isfinite(f) & (f > 0)], HIST_EDGES)[0]
        hist["after"] += np.histogram(z3[np.isfinite(z3) & (z3 > 0)], HIST_EDGES)[0]
        for c in FV_CLASSES:
            m = np.ones(mask.shape, bool) if c == "all" else cls[c]
            removed[c] += np.array([r0[m].sum(), (r0 - r1)[m].sum(), (r1 - r2)[m].sum(), (r2 - r3)[m].sum()]) * FV_STEP_H
    out = {"n_files": len(files), "mask": mask, "hist_edges": HIST_EDGES}
    out.update({f"acc_{k}": v for k, v in acc.items()})
    out.update({f"f55_{k}": v for k, v in f55.items()})
    out.update({f"hist_{k}": v for k, v in hist.items()})
    out.update({f"removed_{k}": v for k, v in removed.items()})
    np.savez_compressed(path, **out)
    return out


fv = first_version(FV_YEAR)
m1 = np.load(DATA_DIR / "mask_BEL_1km.npy")[2 * 1053:2 * 1531, 2 * 528:2 * 1027]
bel = m1.reshape(478, 2, 499, 2).mean((1, 3)) >= 0.5          # crop rows 1053-1531, cols 528-1027 (data.md)
print(f"{int(fv['n_files'])} files; static mask of {FV_YEAR - 1}: {int(fv['mask'].sum()):,} px, {int((fv['mask'] & bel).sum())} in Belgium")
print(f"{'pixels':11s} {'> 70 dBZ':>9s} {'Gabella':>9s} {'static':>9s} {'total':>9s}   (share of the rain removed)")
for c in FV_CLASSES:
    b, a, g, s = fv[f"removed_{c}"]
    print(f"{c:11s} {a / b:9.2%} {g / b:9.2%} {s / b:9.2%} {(a + g + s) / b:9.2%}")
print(f"Belgium: annual sum (index) {fv['acc_before'][bel].mean():,.0f} -> {fv['acc_after'][bel].mean():,.0f} mm, "
      f"{1 - fv['acc_after'][bel].sum() / fv['acc_before'][bel].sum():.1%} removed")
nf = int(fv["n_files"])
f0, f1 = fv["f55_before"] / nf, fv["f55_after"] / nf
print(f"pixels with >= 55 dBZ in more than 0.1% of the times: {int((f0 > 1e-3).sum()):,} -> {int((f1 > 1e-3).sum()):,}; "
      f"more than 1%: {int((f0 > 1e-2).sum())} -> {int((f1 > 1e-2).sum())}")
e = fv["hist_edges"]
for lo, hi in ((0, 20), (20, 40), (40, 55), (55, 70), (70, 131)):
    s_ = (e[:-1] >= lo) & (e[:-1] < hi)
    print(f"echoes {lo:3d}-{hi:3d} dBZ kept: {fv['hist_after'][s_].sum() / max(fv['hist_before'][s_].sum(), 1):6.1%}")

fig, ax = plt.subplots(1, 4, figsize=(24, 5.6), layout="constrained")
vmax = np.percentile(fv["acc_before"], 99)
for i, (k, t) in enumerate((("acc_before", f"ODYSSEY {FV_YEAR}: annual sum before (index, mm)"),
                            ("acc_after", f"after: > 70 dBZ, Gabella, histo_cut of {FV_YEAR - 1}"))):
    im = ax[i].imshow(fv[k], cmap="viridis", vmax=vmax); ax[i].set_title(t)
fig.colorbar(im, ax=ax[:2], shrink=0.8)
for i, (f_, t) in enumerate(((f0, "frequency of >= 55 dBZ, before"), (f1, "after"))):
    im = ax[2 + i].imshow(np.where(f_ > 0, f_, np.nan), cmap="magma_r", norm=LogNorm(vmin=1e-5, vmax=1e-1)); ax[2 + i].set_title(t)
fig.colorbar(im, ax=ax[2:], shrink=0.8)
for a_ in ax:
    a_.set_xticks([]); a_.set_yticks([])
plt.show()

## What is seen

Run on 9/11/2026.

- **Summer convective days:** the filter removes 4.9% of the convective rain at 2 km (4.0–6.3% per day) and
  2.3% at 1 km (1.3–3.0%); 0.9% and 0.6% of the stratiform rain.
- **January days with stationary features:** it removes 11.2% of the convective rain at 2 km (9.2–16.4% per
  day) and 7.1% at 1 km (6.1–10.3%): more than twice the summer share at 2 km, three times at 1 km. These
  "convective" pixels are mostly the stationary features, but not only them, so this is an upper bound of
  the gain on the features, not a measure of it.
- **Where the removed convective rain lies:** 92–93% inside echo regions, 5–7% on edges, 1–3% in small
  echoes. The filter acts mostly through its gradient test, on pixels much stronger than their neighbours,
  not by removing small cells.
- **Catalogue, ODYSSEY 2023 and CIRRUS 2025** (section 4):
  - The annual sums (median 2,831 and 2,204 mm) are far above the rain that falls in the region, because
    a column-maximum reflectivity converted with Marshall–Palmer is not a rain estimate; they serve only as
    a statistic for `histo_cut`.
  - With 5% tails, `histo_cut` flags 8,148 ODYSSEY pixels (3.4% of the crop) and 31,851 CIRRUS pixels
    (3.3%) as static clutter, and no shading. Most flagged areas are discs centred on the radars, where the
    annual sum is highest, plus the patches on and off the Dutch coast; the discs may be the effect of
    distance from the radar on a maximum composite rather than clutter (not checked).
  - The frequency of ≥ 55 dBZ shows the lines over the North Sea in both products, and radial lines in the
    south-east of ODYSSEY.
- **Catalogue, ODYSSEY 2013–2024** (sections 4–5, run on 10/11/2026):
  - `histo_cut` flags 6,042–12,172 pixels a year (8,131 in 2013, 6,042 in the half year 2024) and never any
    shading.
  - **The mask is not stable.** Only 631 pixels are flagged in all 12 years and 1,949 in 10 or more;
    15,748 are flagged in only 1 to 3 years. Of each year's mask, 59–83% was already flagged the year before.
  - The stable pixels are the centres of the discs and the fixed points. The outer rings of the discs change
    from year to year, as rain would. The annual sums show the discs as regularly spaced bright spots, each
    with radial structure, consistent with radar sites (positions not checked): the sum falls with distance
    from each radar. So with 5% tails `histo_cut` mixes clutter with the decrease of the annual sum with
    distance from the radar. Wagner et al. (2012) correct that decrease separately ("beam height").
- **First version on ODYSSEY 2023** (section 6, mask of 2022):
  - Rain removed, all pixels: 0.16% by > 70 dBZ, 5.12% by Gabella, 3.87% by the static mask, 9.15% in
    total. Convective pixels: 0.25%, 7.36%, 5.25%, 12.86%. Stratiform: 0.00%, 1.37%, 1.56%, 2.93%.
  - Over Belgium, 12.0% of the annual sum is removed (3,765 → 3,312 mm, index).
  - Echoes kept: 99.8% of those above 70 dBZ are removed (the 83 left are exactly 70.0 dBZ); 63% of those of
    55–70 dBZ are kept, 85% of 40–55, 94% of 20–40, 93% of 0–20.
  - Pixels with ≥ 55 dBZ in more than 0.1% of the times: 1,984 → 209; more than 1%: 166 → 0. The North Sea
    lines and the fixed points go; part of the radial lines in the south-east stays (interference residual).
  - **The mask of 2022 overshoots on 2023:** it leaves dark rings and holes around several radars in the
    annual sum after cleaning, and blank patches in the ≥ 55 dBZ frequency, where real strong echoes of 2023
    were lowered with the clutter. This follows from the instability of section 5.
  - Gabella removes 7.36% of the convective rain over the year, more than on the five summer days of section
    1 (4.9% at 2 km), because its share includes the fixed points that Steiner classes as convective.